# Deep E-prop: Credit Assignment Across Time and Depth

**Course:** NeuroAI & ML 4 Neuro - Sommersemester 2026
**Authors:** Simon Peter ; Yannick Säckl ; Ruchit Kumar Patel
**Repository:** github.com/simonpeter02/e-prop-in-deep-networks

---

## Abstract

tbd

| Experiment | Question |
|-----------|----------|
| **1 — Single-layer e-prop** | Does e-prop match BPTT on a delayed-recall task (reproduce Bellec 2020)? |
| **2 — Leaky integrators (α sweep)** | When does the temporal eligibility trace actually matter (e-prop vs d=0 across α)? |
| **3 — Time *and* depth (main result)** | Does deep e-prop assign credit across time and depth at once? Full deep e-prop vs ablate-spatial vs ablate-temporal vs BPTT, on a leaky deep RNN doing hierarchical classify-then-count. |

**How to read this notebook:** each experiment below is split into two parts: **(A) Reproduce**, which loads the committed `results/*.json` and replots in seconds with no training, and **(B) Full rerun**, which repeats all the training/gradient computation from scratch and overwrites those JSON files. Run only the Reproduce cells for a quick check, or run the notebook top to bottom to verify everything from scratch. `results/` is no longer wiped at the start of a run (so the Reproduce cells always have something to load); each Full-rerun cell overwrites its own output files in place. Architecture diagrams go to `figures/`. The final cell pushes outputs to GitHub.

---
## 1. Setup

Clone the repository, install dependencies, and auto-detect the compute device (GPU preferred).
All subsequent sections depend on this cell.

In [ ]:
# Clone repo (skip if already present), then pull latest changes
import os

REPO_URL = "https://github.com/simonpeter02/e-prop-in-deep-networks.git"
REPO_DIR = "e-prop-in-deep-networks"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

In [ ]:
!pip install -q torch numpy matplotlib scipy

In [ ]:
%matplotlib inline
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch.nn.functional as F
import sys, os

# Auto-detect best available device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Batch size: larger on GPU so each kernel has enough work to saturate the device
BATCH_GPU  = 128
BATCH_CPU  = 32
BATCH_DEFAULT = BATCH_GPU if DEVICE == "cuda" else BATCH_CPU
print(f"Default batch size: {BATCH_DEFAULT}")

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

In [ ]:
from tasks.store_and_recall  import generate_batch, task_accuracy
from models.vanilla_rnn      import VanillaRNN
from models.leaky_rnn        import LeakyRNN
from learning_rules.eprop    import compute_eprop_gradients, compute_eprop_leaky_gradients, mse_error as sl_mse, xent_error
from learning_rules.bptt     import compute_bptt_gradients, _mse_loss, _xent_loss

SEED       = 42
N_PATTERNS = 4
torch.manual_seed(SEED)
np.random.seed(SEED)

print('Imports OK')

In [ ]:
# MASTER CONFIG

# Exp 1: single-layer e-prop (store-and-recall)
N_REC_SL   = 100
DELAY_SL   = 2
N_STEPS    = 1000
EVAL_EVERY = 50
LR         = 1e-3

# Exp 2: leaky RNN alpha sweep
N_REC_LK     = 100
ALPHAS_LK    = [0.05, 0.1, 0.2, 0.5, 1.0]
DELAY_LK     = 5
N_TRIALS_LK  = 30
ALPHA_LC     = 0.1
LR_LK        = 5e-3

# Batch sizes (set from device; see cell above)
BATCH_SL  = BATCH_DEFAULT
BATCH_LK  = BATCH_DEFAULT

# Input sizes
n_in     = N_PATTERNS + 2   # store-and-recall: patterns + recall + bias
n_in_lk  = N_PATTERNS + 2   # leaky RNN (same task)

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Master config loaded.")

In [ ]:
def save_fig(fig, name, out_dir="results"):
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(f"{out_dir}/{name}.pdf", bbox_inches='tight')
    fig.savefig(f"{out_dir}/{name}.svg", bbox_inches='tight')
    print(f"Saved {out_dir}/{name}.pdf / .svg")

def bptt_grads_deep(model, inputs, targets, mask, loss_fn=None):
    if loss_fn is None:
        loss_fn = _mse_loss
    for p in model.parameters():
        if p.grad is not None: p.grad.zero_()
    outputs, _ = model(inputs)
    loss_fn(outputs, targets, mask).backward()
    return {k: p.grad.clone() for k, p in model.named_parameters() if p.grad is not None}

def cosine_keys(g1, g2, keys):
    sims = []
    for k in keys:
        if k not in g1 or k not in g2: continue
        v1, v2 = g1[k].flatten(), g2[k].flatten()
        if v1.norm() < 1e-12 or v2.norm() < 1e-12: continue
        sims.append(F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item())
    return float(np.mean(sims)) if sims else float('nan')

print("Helpers defined")

In [ ]:
# results/ is intentionally NOT wiped here: the "Reproduce from saved results" cells
# in each experiment below read the JSON already committed to results/, and every
# "Full rerun" cell overwrites its own named output files in place. Delete the
# results/ folder yourself first if you want a byte-for-byte clean regeneration.
import os
os.makedirs("results", exist_ok=True)
print("results/ ready (existing contents kept; see note above).")

---
## 3. Experiment 1 - Single-layer e-prop (Minimal Viable Result #1)

**Question:** On the simplest possible case (one hidden layer, short delay), do e-prop and d=0 match BPTT in both learning speed and gradient direction?

**Setup:** A single-layer leaky tanh RNN is trained on the cue accumulation task with variable delay. We compare two learning rules:

| Rule | Description |
|------|-------------|
| **BPTT** | Exact gradient via PyTorch autograd |
| **E-prop** | Diagonal approximation of RTRL |

**Prediction:** ...

#### A. Reproduce from saved results (fast, no training)

Loads `results/exp1_*.json` and replots, without training. **Not available yet:**
Experiment 1 (Ruchit's single-layer e-prop reproduction of Bellec et al. 2020) hasn't
been added to the repository yet (see `README.md` §2/§4). The cell below checks for
the results anyway and reports what's missing, so it's safe to run even before that
code lands.

In [ ]:
# ── Experiment 1 — reproduce from saved results ───────────────────────────────────
import glob, json as _json

_exp1_json = sorted(glob.glob("results/exp1_*.json"))
if not _exp1_json:
    print("No results/exp1_*.json found: Experiment 1's training code hasn't been added "
          "to the repository yet (see README.md). Nothing to reproduce here yet — run "
          "part B once that code lands; it should also save its metrics to "
          "results/exp1_<description>.json so this cell can replot them without retraining.")
else:
    for _path in _exp1_json:
        with open(_path) as _f:
            _data = _json.load(_f)
        print(f"Loaded {_path}: keys={list(_data.keys())}")
    # TODO: once Experiment 1 saves its JSON, replot its figure(s) here the same way
    # the Experiment 2 "Reproduce" cells below do (load -> matplotlib -> save_fig).

#### B. Full rerun (trains from scratch)

Trains Experiment 1 end to end and saves its figures + results JSON. **Pending** —
see part A above.

In [ ]:
# TODO (Experiment 1 — single-layer e-prop reproduction of Bellec et al. 2020):
# train + evaluate on tasks.store_and_recall (see experiments/single_layer_eprop.py
# for a starting point), then save the metrics to results/exp1_<description>.json so
# the "Reproduce" cell above can replot them without retraining.
print("Experiment 1 is not implemented in this notebook yet - see README.md. Skipping "
      "so the rest of the notebook can still run top to bottom.")

---
## 5. Experiment 2 - Credit assignment across TIME and DEPTH, simultaneously (main result)

**Question.** Millidge (2025) shows deep e-prop *should* assign credit across both
depth and time. Does it, in practice, *at the same time*?

**Setup.** A 2-layer **leaky** `DeepRNN` with a FAST lower layer (transient feature
extractor, α=0.5) and a SLOW top layer (integrator, α=0.05), trained on a
hierarchical **classify-then-count** task (`tasks/hierarchical_cue.py`): each cue is
a mean-zero rising/falling temporal motif. The lower layer must learn the temporal
feature (**depth**); the top layer must accumulate the per-cue results across a
silent delay (**time**). 

**Two controls on the top-layer trace** `ε^z = (∂z/∂h)·ε^h + (∂z/∂z₍ₜ₋₁₎)·ε^z₍ₜ₋₁₎`:
- `ablate_spatial` (∂z/∂h = 0): removes the **depth** path → lower-layer grads → 0.
- `ablate_temporal` (∂z/∂z₍ₜ₋₁₎ = 0): removes the **cross-layer temporal** path.

**Third control — readout-only (reservoir)** (5.4): both recurrent layers stay
FROZEN at random init; only `W_out`/`b_out` train (the e-prop readout gradient is
exact). This is a 2-layer random reservoir + linear readout, and it is the natural
**floor** for `ablate_spatial`: that ablation freezes layer 0 but still trains
layer 1, so `ablate_spatial − readout-only` measures what training the top layer
buys over a random reservoir — i.e. whether depth credit is actually necessary.

**Predictions.** (5.1) full deep e-prop aligns with BPTT for *both* layers, while
each control breaks one dimension; most of the lower-layer credit flows through the
cross-layer temporal trace ε^z. (5.2) **BPTT ≥ full e-prop > both controls** in
learning. The full multi-seed study (incl. a delay sweep, E3) lives in
`experiments/deep_credit_time_depth.py`; the cells below run a compact version.

**This section runs in two parts:** **(A) Reproduce** regenerates every figure below
from the committed JSON in `results/` — fast, no training. **(B) Full rerun** repeats
all the training and gradient computation from scratch and overwrites those JSON
files — slower; use it to verify the numbers.

### A. Reproduce from saved results (fast, no training, ~seconds)

Every cell below just loads the JSON already committed in `results/` and replots — no
model is instantiated and nothing is trained.

| Sub-result | Figure(s) | Source JSON |
|---|---|---|
| 5.1 Gradient credit | `exp5_gradient_credit`, `exp5_credit_summary` | `exp5_gradient_credit.json` |
| 5.1b Cue decoding | `exp5_cue_decoding` | `exp5_cue_decoding.json` |
| 5.1c Gradient geometry (PCA) | `exp5_gradient_geometry` | `exp5_gradient_geometry.json` |
| 5.2 Adam learning curves | `exp5_learning_curves` | `exp5_learning_curves.json` |
| 5.3 Speed significance | `exp5_speed_threshold`, `exp5_speed_intervals` | `exp5_learning_curves.json` (per-seed curves; significance is recomputed on saved data, nothing is retrained) |
| 5.4 Reservoir control | `exp5_reservoir_control`, `exp5_reservoir_curves` | `exp5_reservoir_control.json` + `exp5_learning_curves.json` |

In [ ]:
# ── Reproduce from saved results: setup ─────────────────────────────────────
import json

def load(name, out_dir="results"):
    with open(f"{out_dir}/{name}.json") as f:
        return json.load(f)

# Shared style/labels (mirrors the Full-rerun setup in 5. Experiment 2 setup below)
col = {"bptt": "k", "full": "C0", "ablate_temporal": "C3", "ablate_spatial": "C1", "readout": "C7"}
e5_methods = [("bptt", "BPTT"), ("full", "deep e-prop (full)"),
              ("ablate_temporal", "ablate temporal"), ("ablate_spatial", "ablate spatial")]
_lab = {m: lab for m, lab in e5_methods}
E5_DELAY_MAIN = 12     # delay shown in the credit-summary bars (5.1) and reservoir bars (5.4)
E5_THETA = 0.9         # accuracy threshold for the 5.3 speed-to-threshold figure
E5B_MODES = ["full", "ablate_temporal", "ablate_spatial"]
print("Reproduce setup ready.")

In [ ]:
# ── 5.1 (reproduce) — per-layer cosine vs delay + cross-temporal share + summary bars ──
gc = load("exp5_gradient_credit")
DELAYS = gc["delays"]
e5_mean, e5_sem = gc["mean"], gc["sem"]   # keyed by str(delay) -> {full_low, ...}

def _ser(stat, k):
    return np.nan_to_num(np.array([stat[str(d)][k] for d in DELAYS], dtype=float))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
for k, c, ls, lab, mfc in [("full_up", "C0", ":", "full · output-adjacent", "none"),
                           ("full_low", "C0", "-", "full · input-adjacent", "C0"),
                           ("temp_low", "C3", "-", "ablate_temporal · input-adjacent", "C3"),
                           ("spat_low", "C1", "-", "ablate_spatial · input-adjacent (=0)", "C1")]:
    mu, er = _ser(e5_mean, k), _ser(e5_sem, k)
    ax.plot(DELAYS, mu, marker="o", color=c, ls=ls, label=lab, markerfacecolor=mfc)
    ax.fill_between(DELAYS, mu - er, mu + er, color=c, alpha=0.15)
ax.axhline(0, color="gray", ls=":"); ax.set_xlabel("delay D"); ax.set_ylabel("cosine vs BPTT")
ax.set_ylim(-0.1, 1.05); ax.legend(fontsize=8)
ax = axes[1]
mu, er = _ser(e5_mean, "xshare"), _ser(e5_sem, "xshare")
ax.plot(DELAYS, mu, "o-", color="C2"); ax.fill_between(DELAYS, mu - er, mu + er, color="C2", alpha=0.15)
ax.set_xlabel("delay D"); ax.set_ylabel("||full - ablate_temporal|| / ||full||"); ax.set_ylim(0, 1.05)
fig.tight_layout(); save_fig(fig, "exp5_gradient_credit"); plt.show()

db = E5_DELAY_MAIN if str(E5_DELAY_MAIN) in e5_mean else DELAYS[len(DELAYS) // 2]
kf = {("full", "low"): "full_low", ("full", "top"): "full_up",
      ("ablate_temporal", "low"): "temp_low", ("ablate_temporal", "top"): "temp_up",
      ("ablate_spatial", "low"): "spat_low", ("ablate_spatial", "top"): "spat_up"}
fig, ax = plt.subplots(figsize=(7, 4.5)); x = np.arange(2); w = 0.26
for i, (meth, c) in enumerate([("full", "C0"), ("ablate_temporal", "C3"), ("ablate_spatial", "C1")]):
    mus = [np.nan_to_num(e5_mean[str(db)][kf[(meth, l)]]) for l in ("low", "top")]
    ers = [np.nan_to_num(e5_sem[str(db)][kf[(meth, l)]]) for l in ("low", "top")]
    ax.bar(x + (i - 1) * w, mus, w, yerr=ers, capsize=4, color=c, label=meth.replace("ablate_", "ablate "))
ax.set_xticks(x); ax.set_xticklabels(["input-adjacent", "output-adjacent"]); ax.set_ylim(-0.05, 1.05)
ax.axhline(0, color="gray", ls=":"); ax.set_ylabel("cosine vs BPTT")
ax.legend(fontsize=8); fig.tight_layout(); save_fig(fig, "exp5_credit_summary"); plt.show()
print(f"Reproduced 5.1 from results/exp5_gradient_credit.json (n={gc.get('n')} seeds/delay).")

In [ ]:
# ── 5.1b (reproduce) — cue decoding: temporal carry makes lower-layer credit cue-specific ──
cd = load("exp5_cue_decoding")
e5b_sum = cd["summary"]
E5B_COLOR = {"full": "C0", "ablate_temporal": "C3", "ablate_spatial": "C1"}
_TG = [("label", "binary label\n(readout-sign leak)"), ("margin", "cue margin\n(unanimous vs split)")]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, lyr, ttl in [(axes[0], "lower", "input-adjacent (lower) layer"),
                     (axes[1], "upper", "output-adjacent (top) layer")]:
    x = np.arange(len(_TG)); w = 0.26
    for i, mode in enumerate(E5B_MODES):
        mus = [e5b_sum[lyr][mode][t]["acc"] for t, _ in _TG]
        ers = [e5b_sum[lyr][mode][t]["sem"] for t, _ in _TG]
        ax.bar(x + (i - 1) * w, mus, w, yerr=ers, capsize=4, color=E5B_COLOR[mode],
               label=mode.replace("ablate_", "ablate "))
    chi = np.mean([e5b_sum[lyr][mode][t]["chance_hi"] for mode in E5B_MODES for t, _ in _TG])
    ax.axhline(0.5, color="gray", ls="--", lw=1)
    ax.axhline(chi, color="gray", ls=":", lw=1, label="perm. 95%")
    ax.set_xticks(x); ax.set_xticklabels([lab for _, lab in _TG])
    ax.set_title(ttl); ax.set_ylim(0.4, 1.02)
    if lyr == "lower":
        ax.set_ylabel("cross-validated decode accuracy"); ax.legend(fontsize=8, loc="upper right")
fig.suptitle("Cue information in the per-trial gradient: the temporal carry makes "
             "lower-layer credit cue-specific", fontsize=11)
fig.tight_layout(); save_fig(fig, "exp5_cue_decoding"); plt.show()
print(f"Reproduced 5.1b from results/exp5_cue_decoding.json "
      f"(seeds={cd['seeds']}, trials={cd['trials']}, delay={cd['delay']}).")

In [ ]:
# ── 5.1c (reproduce) — gradient geometry (PCA), colored by cue type ────────────
gg = load("exp5_gradient_geometry")
_count0 = np.array(gg["count"])
_ncues = int(_count0.max())
_ccolors = {c: plt.cm.viridis(c / _ncues) for c in range(_ncues + 1)}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, mode in zip(axes, gg["modes"]):
    entry = gg["per_mode"].get(mode)
    mode_lab = mode.replace("ablate_", "ablate ")
    if entry is None:
        ax.text(0.5, 0.5, "gradient ≡ 0\n(ablate_spatial removes\nlower-layer credit)",
                ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(mode_lab, fontsize=10)
    else:
        proj = np.array(entry["proj"]); var = entry["explained_variance_ratio"]; acc = entry["margin_decode_acc"]
        for c in range(_ncues + 1):
            sel = _count0 == c
            if not sel.any():
                continue
            ax.scatter(proj[sel, 0], proj[sel, 1], s=14, alpha=0.7,
                       color=_ccolors[c], label=f"{c} rising")
        ax.set_xlabel(f"PC1 ({var[0]*100:.0f}%)")
        ax.set_ylabel(f"PC2 ({var[1]*100:.0f}%)")
        ax.set_title(f"{mode_lab}\ndecode acc (margin) = {acc:.2f}", fontsize=10)
axes[0].legend(fontsize=8, loc="best", title="rising cues")
fig.suptitle("Geometry of the per-trial lower-layer gradient (PCA, seed 0), colored by "
             "cue type (rising-cue count): the temporal carry makes it cue-specific", fontsize=11)
fig.tight_layout(); save_fig(fig, "exp5_gradient_geometry"); plt.show()
print(f"Reproduced 5.1c from results/exp5_gradient_geometry.json (seed={gg['seed']}).")

In [ ]:
# ── 5.2 (reproduce) — Adam learning curves: credit quality sets convergence speed ───
lc = load("exp5_learning_curves")
E5_EVAL = lc["eval_every"]; E5_LR = lc["lr"]
e5_curves = {m: (np.array(v[0]), np.array(v[1])) for m, v in lc["curves"].items()}
e5_plot_methods = e5_methods + [("readout", "readout-only (reservoir)")]

fig, ax = plt.subplots(figsize=(7, 4.5))
for meth, lab in e5_plot_methods:
    mu, er = e5_curves[meth]
    x = np.arange(len(mu)) * E5_EVAL
    ls = "--o" if meth == "readout" else "-o"
    ax.plot(x, mu, ls, ms=3, color=col[meth], label=lab)
    ax.fill_between(x, mu - er, mu + er, color=col[meth], alpha=0.15)
ax.axhline(0.5, color="gray", ls="--", label="chance")
ax.set_xlabel("training step"); ax.set_ylabel("accuracy (held-out)"); ax.set_ylim(0.45, 1.02)
ax.set_title(f"Adam (lr={E5_LR:.0e}): trainable rules converge equal; reservoir floors below")
ax.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "exp5_learning_curves"); plt.show()
print(f"Reproduced 5.2 from results/exp5_learning_curves.json "
      f"(eval_every={E5_EVAL}, optimizer={lc.get('optimizer')}, lr={E5_LR:.0e}).")

In [ ]:
# ── 5.3 (reproduce) — speed significance: steps-to-threshold + cluster permutation ───
from experiments.stats import paired_report, holm, cluster_perm_test

lc = load("exp5_learning_curves")
E5_EVAL = lc["eval_every"]
e5_rows = {m: lc["per_seed"][m] for m, _ in e5_methods}   # ragged per-seed eval curves

E5_SPEED_COMPS = [("full", "ablate_temporal", "full vs ablate_temporal", "C4"),
                  ("full", "ablate_spatial",  "full vs ablate_spatial",  "C5"),
                  ("ablate_temporal", "ablate_spatial", "temporal vs spatial", "C6")]

def _stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

def steps_to_threshold(r, theta, eval_every=E5_EVAL):
    r = np.asarray(r, float)
    above = np.where(r >= theta)[0]
    if len(above) == 0:
        return float("nan")
    i = int(above[0])
    if i == 0:
        return 0.0
    y0, y1 = r[i - 1], r[i]
    frac = 0.0 if y1 == y0 else (theta - y0) / (y1 - y0)
    return (i - 1 + min(max(frac, 0.0), 1.0)) * eval_every

s2t = {m: np.array([steps_to_threshold(r, E5_THETA) for r in e5_rows[m]]) for m, _ in e5_methods}
reports = [(lab, a, b, paired_report(s2t[a], s2t[b])) for a, b, lab, _ in E5_SPEED_COMPS]
e5_spd_padj = holm([r["p_perm"] for *_, r in reports])
print(f"Steps to reach acc>={E5_THETA:.2f} (paired sign-flip perm, Holm):")
for (lab, a, b, r), pa in zip(reports, e5_spd_padj):
    print(f"  {lab:26s} d_steps={r['mean_diff']:+7.0f}  dz={r['cohen_dz']:+5.2f}  "
          f"p={r['p_perm']:.4f}  Holm={pa:.4f}  {_stars(pa)}")

order = ["bptt", "full", "ablate_temporal", "ablate_spatial"]
short = {"bptt": "BPTT", "full": "full", "ablate_temporal": "ablate\ntemporal",
         "ablate_spatial": "ablate\nspatial"}
means = {m: float(np.nanmean(s2t[m])) for m in order}
sems  = {m: float(np.nanstd(s2t[m], ddof=1) / np.sqrt(np.sum(~np.isnan(s2t[m])))) for m in order}
figA, axA = plt.subplots(figsize=(7, 4.6)); xs = np.arange(len(order))
axA.bar(xs, [means[m] for m in order], yerr=[sems[m] for m in order], capsize=4,
        color=[col[m] for m in order])
axA.set_xticks(xs); axA.set_xticklabels([short[m] for m in order])
axA.set_ylabel(f"training steps to reach acc >= {E5_THETA:.2f}")
axA.set_title(f"Convergence speed (Adam): steps to {E5_THETA:.0%} accuracy")
y0 = max(means[m] + sems[m] for m in order)
for k, (a, b, lab, _) in enumerate(E5_SPEED_COMPS):
    xa, xb = order.index(a), order.index(b); y = y0 * (1.06 + 0.10 * k)
    axA.plot([xa, xa, xb, xb], [y - y0 * 0.02, y, y, y - y0 * 0.02], color="k", lw=1)
    axA.text((xa + xb) / 2, y, _stars(e5_spd_padj[k]), ha="center", va="bottom", fontsize=11)
axA.set_ylim(0, y0 * 1.45)
figA.tight_layout(); save_fig(figA, "exp5_speed_threshold"); plt.show()

n_common = min(len(r) for m, _ in e5_methods for r in e5_rows[m])
A = {m: np.array([r[:n_common] for r in e5_rows[m]]) for m, _ in e5_methods}
t_axis = np.arange(n_common) * E5_EVAL

e5_sig_intervals = {}
print(f"\nCluster-permutation significant intervals (0-{t_axis[-1]} steps):")
for a, b, lab, _c in E5_SPEED_COMPS:
    cl = cluster_perm_test(A[a] - A[b])
    sig = [c for c in cl if c["p_cluster"] < 0.05]
    e5_sig_intervals[(a, b)] = sig
    spans = ", ".join(f"[{t_axis[c['t0']]}-{t_axis[c['t1']]}] (p={c['p_cluster']:.3f})"
                      for c in sig) or "none"
    print(f"  {lab:26s} {spans}")

figB, axB = plt.subplots(figsize=(7, 4.8))
for meth, lab in e5_methods:
    axB.plot(t_axis, A[meth].mean(0), "-o", ms=3, color=col[meth], label=lab)
axB.axhline(0.5, color="gray", ls="--", lw=0.8)
axB.set_xlabel("training step"); axB.set_ylabel("accuracy (held-out)")
axB.set_ylim(0.40, 1.02)
axB.set_title("Where convergence-speed gaps are significant (cluster permutation)")
ybar = 0.46
for k, (a, b, lab, cc) in enumerate(E5_SPEED_COMPS):
    y = ybar - 0.018 * k
    for c in e5_sig_intervals[(a, b)]:
        axB.plot([t_axis[c["t0"]], t_axis[c["t1"]]], [y, y], color=cc, lw=4, solid_capstyle="butt")
    axB.plot([], [], color=cc, lw=4, label=f"sig: {lab}")
axB.legend(fontsize=7, ncol=2, loc="lower right")
figB.tight_layout(); save_fig(figB, "exp5_speed_intervals"); plt.show()
print("Reproduced 5.3 from results/exp5_learning_curves.json (per-seed curves; "
      "significance recomputed on saved data, nothing retrained).")

In [ ]:
# ── 5.4 (reproduce) — readout-only reservoir control ─────────────────────
MINUS = chr(8722)  # U+2212, as used in the saved comparison labels
rc = load("exp5_reservoir_control")
lc = load("exp5_learning_curves")
e5_curves = {m: (np.array(v[0]), np.array(v[1])) for m, v in lc["curves"].items()}

means, sems = rc["means"], rc["sems"]
bar_methods = [("bptt", "BPTT"), ("full", "full"), ("ablate_temporal", "ablate\ntemporal"),
               ("ablate_spatial", "ablate\nspatial"), ("readout", "readout-only\n(reservoir)")]

figc, axc = plt.subplots(figsize=(7.5, 4.6)); xs = np.arange(len(bar_methods))
axc.bar(xs, [means[m] for m, _ in bar_methods], yerr=[sems[m] for m, _ in bar_methods],
        capsize=4, color=[col[m] for m, _ in bar_methods])
axc.set_xticks(xs); axc.set_xticklabels([lab for _, lab in bar_methods])
axc.set_ylabel("converged accuracy (held-out)"); axc.axhline(0.5, color="gray", ls="--", label="chance")
axc.set_ylim(0.45, 1.05); axc.legend(fontsize=8)
axc.set_title("Depth ablation vs a random reservoir: does training the top layer rescue it?")
figc.tight_layout(); save_fig(figc, "exp5_reservoir_control"); plt.show()

res_mu, res_sem = np.array(rc["res_curve"][0]), np.array(rc["res_curve"][1])
figd, axd = plt.subplots(figsize=(7, 4.5))
for meth, lab in [("bptt", "BPTT"), ("full", "full"),
                  ("ablate_temporal", "ablate temporal"), ("ablate_spatial", "ablate spatial")]:
    mu, er = e5_curves[meth]; x = np.arange(len(mu)) * lc["eval_every"]
    axd.plot(x, mu, "-o", ms=3, color=col[meth], label=lab)
    axd.fill_between(x, mu - er, mu + er, color=col[meth], alpha=0.12)
xr = np.arange(len(res_mu)) * lc["eval_every"]
axd.plot(xr, res_mu, "--o", ms=3, color=col["readout"], label="readout-only (reservoir)")
axd.fill_between(xr, res_mu - res_sem, res_mu + res_sem, color=col["readout"], alpha=0.12)
axd.axhline(0.5, color="gray", ls="--", label="chance")
axd.set_xlabel("training step"); axd.set_ylabel("accuracy (held-out)"); axd.set_ylim(0.45, 1.02)
axd.legend(fontsize=8); axd.set_title("Reservoir floor vs trainable-hidden rules")
figd.tight_layout(); save_fig(figd, "exp5_reservoir_curves"); plt.show()

print(f"  {'comparison':30s} {'meanD':>8} {'dz':>7} {'perm p':>8} {'Holm p':>8}  sig")
for r in rc["comparisons"]:
    pa = r["p_holm"]
    sig = "***" if pa < 0.001 else "**" if pa < 0.01 else "*" if pa < 0.05 else "ns"
    print(f"  {r['comparison'].replace(MINUS, '-'):30s} {r['mean_diff']:+8.3f} "
          f"{r['cohen_dz']:+7.2f} {r['p_perm']:8.4f} {pa:8.4f}  {sig}")
print(f"Reproduced 5.4 from results/exp5_reservoir_control.json + exp5_learning_curves.json "
      f"(delay={rc['delay']}, n_seeds={rc['n_seeds']}).")

### B. Full rerun (trains everything from scratch)

Everything below repeats the training and gradient computation described above and
overwrites the JSON files that part A reads. It reruns 5.1's gradient-cosine sweep,
5.1b/5.1c's per-trial-gradient decoding and PCA, 5.2's Adam training runs (incl. the
reservoir control), and recomputes 5.3/5.4's significance tests on the freshly
trained curves. Expect this to take noticeably longer than part A — minutes rather
than seconds, more on CPU than GPU.

In [ ]:
# ── Experiment 5 setup ──────────────────────────────────────────────────────
from models.deep_rnn          import DeepRNN
from learning_rules.deep_eprop import compute_deep_eprop_gradients, xent_error
from learning_rules.bptt       import compute_bptt_gradients, _xent_loss
from learning_rules.interface  import apply_gradients
from tasks.hierarchical_cue    import generate_batch as hcue, task_accuracy as hacc
from utils                     import cosine_sim_grads, flat_grads
import time, json, warnings

E5_NREC   = 32
E5_ALPHA  = [0.5, 0.05]          # fast lower extractor + slow top integrator
E5_NCUES  = 3
E5_TASK   = dict(cue_duration=3, inter_cue_interval=2, amp=2.0, feature_noise=0.15)
E5_LOWER  = ["W_in", "W_recs.0", "biases.0"]      # lower-layer (layer-0) params
E5_UPPER  = ["W_recs.1", "W_ffs.0", "biases.1"]   # top-layer (layer-1) params
E5_DELAYS     = [6, 12, 20]
E5_DELAY_MAIN = 12               # learning-curve delay; also the summary-bar delay
E5_STEPS      = 2000             # ↑ (legacy fixed budget; superseded by E5_MAX_STEPS)
E5_SEEDS      = 8                # learning-curve seeds
E5_COS_SEEDS  = 16               # gradient-cosine seeds  ↑
E5_BATCH      = globals().get("BATCH_DEFAULT", 48)
E5_LR         = 1e-3             # Adam learning rate (5.2 trains with Adam — see below)
E5_EVAL       = 100
E5_EVAL_N     = 512              # samples per eval batch (×reps) — de-noises curves
E5_EVAL_REPS  = 4

# ── Optimiser + convergence-stopping config (used in 5.2) ────────────────────
# 5.2 trains with Adam (per-parameter step normalisation), NOT plain SGD. The credit
# ablations distort gradient *magnitude* far more than *direction* (ablate_temporal
# collapses the lower-layer gradient ~14x but keeps cosine≈0.98; ablate_spatial
# zeroes it). Under shared-LR SGD that magnitude deficit masquerades as a lower final
# accuracy; Adam removes magnitude as a variable per synapse (and is the optimiser
# used in the e-prop literature and our SHD scripts). With magnitude neutralised all
# rules converge to the SAME accuracy on this task — the credit-quality difference is
# in convergence SPEED, which the learning curves show. Each method trains to its own
# convergence (patience, with a min-steps floor so a slow starter isn't cut off early).
E5_GRAD_CLIP = 1.0               # global-norm gradient clip (matches SHD scripts)
E5_MIN_STEPS = 1500              # don't allow early stop before this (slow-starter guard)
E5_MAX_STEPS = 4000              # convergence cap / runaway guard
E5_PATIENCE  = 8                 # evals (×E5_EVAL steps) of no EMA improvement before stopping
E5_TOL       = 5e-3              # min EMA improvement that counts as progress
E5_TAIL_K    = 5                 # evals tail-averaged for the converged-accuracy readout

def e5_batch(B, delay, seed):
    inp, tgt, msk = hcue(B, n_cues=E5_NCUES, delay=delay, seed=seed, **E5_TASK)
    return inp.to(DEVICE), tgt.to(DEVICE), msk.to(DEVICE)

def e5_stats(res):
    # mean and standard error over seeds (rows); all-nan cols (e.g. the unrun tail of
    # an early-stopped seed) return nan/0 without noisy warnings.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        mean = np.nanmean(res, axis=0); sd = np.nanstd(res, axis=0, ddof=1)
    n = np.sum(~np.isnan(res), axis=0)
    sem = np.where(n > 1, sd / np.sqrt(np.maximum(n, 1)), 0.0)
    return mean, sem

print("Exp 5 ready. alpha", E5_ALPHA, "| seeds: cosine", E5_COS_SEEDS, "learning", E5_SEEDS)


In [ ]:
# ── 5.1  Gradient credit: per-layer cosine vs BPTT + cross-temporal share ────
print(f"5.1 gradient credit ({E5_COS_SEEDS} seeds) ...")
E5_CK = ["full_low", "full_up", "temp_low", "temp_up", "spat_low", "spat_up", "xshare"]
e5_mean, e5_sem = {}, {}
for d in E5_DELAYS:
    rows = []
    for s in range(E5_COS_SEEDS):
        torch.manual_seed(1000 + s)
        m = DeepRNN(5, E5_NREC, 2, n_layers=2, alpha=E5_ALPHA).to(DEVICE)
        inp, tgt, msk = e5_batch(E5_BATCH, d, 5000 + s)
        gb = compute_bptt_gradients(m, inp, tgt, msk, _xent_loss)
        gf = compute_deep_eprop_gradients(m, inp, tgt, msk, xent_error, mode="full")
        gt = compute_deep_eprop_gradients(m, inp, tgt, msk, xent_error, mode="ablate_temporal")
        gs = compute_deep_eprop_gradients(m, inp, tgt, msk, xent_error, mode="ablate_spatial")
        a, b = flat_grads(gf, E5_LOWER), flat_grads(gt, E5_LOWER)
        rows.append([cosine_sim_grads(gf, gb, E5_LOWER), cosine_sim_grads(gf, gb, E5_UPPER),
                     cosine_sim_grads(gt, gb, E5_LOWER), cosine_sim_grads(gt, gb, E5_UPPER),
                     cosine_sim_grads(gs, gb, E5_LOWER), cosine_sim_grads(gs, gb, E5_UPPER),
                     ((a - b).norm() / (a.norm() + 1e-12)).item()])
    mu, er = e5_stats(np.array(rows, dtype=float))
    e5_mean[d] = dict(zip(E5_CK, mu)); e5_sem[d] = dict(zip(E5_CK, er))
    print(f"  D={d:3d}: full low={e5_mean[d]['full_low']:.3f}±{e5_sem[d]['full_low']:.3f} "
          f"top={e5_mean[d]['full_up']:.3f} | temporal low={e5_mean[d]['temp_low']:.3f}±{e5_sem[d]['temp_low']:.3f}"
          f" | spatial low=0 | xshare={e5_mean[d]['xshare']:.2f}")

def _ser(stat, k):
    return np.nan_to_num(np.array([stat[d][k] for d in E5_DELAYS]))

# Figure 1 — per-layer cosine vs delay, with stderr bands
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
for k, c, ls, lab, mfc in [("full_up", "C0", ":", "full · output-adjacent", "none"), ("full_low", "C0", "-", "full · input-adjacent", "C0"),
                      ("temp_low", "C3", "-", "ablate_temporal · input-adjacent", "C3"),
                      ("spat_low", "C1", "-", "ablate_spatial · input-adjacent (=0)", "C1")]:
    mu, er = _ser(e5_mean, k), _ser(e5_sem, k)
    ax.plot(E5_DELAYS, mu, marker="o", color=c, ls=ls, label=lab, markerfacecolor=mfc)
    ax.fill_between(E5_DELAYS, mu - er, mu + er, color=c, alpha=0.15)
ax.axhline(0, color="gray", ls=":"); ax.set_xlabel("delay D"); ax.set_ylabel("cosine vs BPTT")
ax.set_ylim(-0.1, 1.05); ax.legend(fontsize=8)
ax = axes[1]
mu, er = _ser(e5_mean, "xshare"), _ser(e5_sem, "xshare")
ax.plot(E5_DELAYS, mu, "o-", color="C2"); ax.fill_between(E5_DELAYS, mu - er, mu + er, color="C2", alpha=0.15)
ax.set_xlabel("delay D"); ax.set_ylabel("||full - ablate_temporal|| / ||full||"); ax.set_ylim(0, 1.05)
fig.tight_layout(); save_fig(fig, "exp5_gradient_credit"); plt.show()

# Figure 2 — summary bars at the main delay (reuses the same cosines)
db = E5_DELAY_MAIN if E5_DELAY_MAIN in e5_mean else E5_DELAYS[len(E5_DELAYS) // 2]
kf = {("full", "low"): "full_low", ("full", "top"): "full_up",
      ("ablate_temporal", "low"): "temp_low", ("ablate_temporal", "top"): "temp_up",
      ("ablate_spatial", "low"): "spat_low", ("ablate_spatial", "top"): "spat_up"}
fig, ax = plt.subplots(figsize=(7, 4.5)); x = np.arange(2); w = 0.26
for i, (meth, c) in enumerate([("full", "C0"), ("ablate_temporal", "C3"), ("ablate_spatial", "C1")]):
    mus = [np.nan_to_num(e5_mean[db][kf[(meth, l)]]) for l in ("low", "top")]
    ers = [np.nan_to_num(e5_sem[db][kf[(meth, l)]]) for l in ("low", "top")]
    ax.bar(x + (i - 1) * w, mus, w, yerr=ers, capsize=4, color=c, label=meth.replace("ablate_", "ablate "))
ax.set_xticks(x); ax.set_xticklabels(["input-adjacent", "output-adjacent"]); ax.set_ylim(-0.05, 1.05)
ax.axhline(0, color="gray", ls=":"); ax.set_ylabel("cosine vs BPTT")
ax.legend(fontsize=8); fig.tight_layout(); save_fig(fig, "exp5_credit_summary"); plt.show()

with open("results/exp5_gradient_credit.json", "w") as _f:
    json.dump({"delays": E5_DELAYS, "mean": e5_mean, "sem": e5_sem, "n": E5_COS_SEEDS}, _f)
print("Saved exp5 gradient credit + summary.")

### 5.1b  Cue decoding — the *temporal carry* is what makes lower-layer credit **cue-specific**

The per-trial lower-layer gradient is $\delta\cdot\varepsilon^z$. The readout error
$\delta=\mathrm{softmax}(o)-y$ flips sign with the **binary label**, so the label is
*trivially* decodable from the per-trial gradient even when the gradient carries **no cue
evidence** — it is just $\delta$'s sign. The real test of cue credit is therefore a
**readout-sign-independent** variable: the cue **margin** (unanimous 3–0 vs split 2–1,
i.e. rising-count $\in\{0,3\}$ vs $\{1,2\}$), which $\delta$'s sign cannot encode.

- **spatial seed** $(\partial z/\partial h)$ = **depth** carry — `ablate_spatial` zeros the lower gradient → decoding falls to chance for *both* targets.
- **temporal carry** $(\partial z/\partial z_{t-1})$ = cross-layer **time** carry — `ablate_temporal` keeps a small lower gradient from the decision-step depth path; the *top* slice of cue-margin credit is lost, so lower-margin decoding drops toward chance while the label (sign) survives.
- **Top layer** (output-adjacent) uses the *self*-trace $\varepsilon^h$, untouched by the $\varepsilon^z$ ablations → decoding is **identical** across all three modes.

In [ ]:
# ── 5.1b  Cue decoding: temporal carry makes lower-layer credit CUE-SPECIFIC ──
print("5.1b cue decoding ...")
E5B_SEEDS, E5B_TRIALS, E5B_KFOLD, E5B_NPERM = 5, 400, 5, 100
E5B_MODES = ["full", "ablate_temporal", "ablate_spatial"]
E5B_COLOR = {"full": "C0", "ablate_temporal": "C3", "ablate_spatial": "C1"}
_CUE_DUR = E5_TASK["cue_duration"]                     # 3
_CUE_STR = _CUE_DUR + E5_TASK["inter_cue_interval"]    # 5

def _cue_vars(inp):
    """Per-trial (binary label, cue margin, rising-count) from the feature channel (ch 0 ramp dir)."""
    x0 = inp[:, :, 0]
    cnt = torch.zeros(inp.shape[1], device=inp.device)
    for c in range(E5_NCUES):
        t0 = c * _CUE_STR
        cnt += (x0[t0 + _CUE_DUR - 1] > x0[t0]).float()      # end>start => rising
    cnt = cnt.cpu().numpy().astype(int)
    label  = (cnt < E5_NCUES / 2.0).astype(int)              # falling-majority = 1
    margin = np.isin(cnt, [0, E5_NCUES]).astype(int)         # unanimous = 1
    return label, margin, cnt

def _per_trial_grads(m, inp, tgt, msk, mode):
    """(N,D) per-trial flattened grads for the lower and upper param groups (B=1 loop)."""
    xl, xu = [], []
    for i in range(inp.shape[1]):
        g = compute_deep_eprop_gradients(m, inp[:, i:i+1], tgt[:, i:i+1], msk[:, i:i+1],
                                         xent_error, mode=mode)
        xl.append(flat_grads(g, E5_LOWER).cpu().numpy())
        xu.append(flat_grads(g, E5_UPPER).cpu().numpy())
    return np.asarray(xl), np.asarray(xu)

# dependency-light linear decoder (sklearn not installed) ----------------------
def _folds(y, k, rng):
    f = [[] for _ in range(k)]
    for cls in np.unique(y):
        ix = np.where(y == cls)[0]; rng.shuffle(ix)
        for j, v in enumerate(ix): f[j % k].append(v)
    return [np.array(sorted(z)) for z in f]

def _bacc(yt, yp):
    return float(np.mean([(yp[yt == c] == c).mean() for c in np.unique(yt)]))

def _ridge(Xtr, ypm, Xte, reg=1.0):
    """Linear-kernel dual ridge (N<<D). ypm in {-1,+1} -> +-1 predictions."""
    Xtr = torch.as_tensor(Xtr, dtype=torch.float32); Xte = torch.as_tensor(Xte, dtype=torch.float32)
    y = torch.as_tensor(ypm, dtype=torch.float32); n = Xtr.shape[0]
    K = Xtr @ Xtr.T; lam = reg * (K.diagonal().mean() + 1e-8)
    a = torch.linalg.solve(K + lam * torch.eye(n), y)
    return torch.sign((Xte @ Xtr.T) @ a).cpu().numpy()

def _decode(X, y, k=E5B_KFOLD, seed=0):
    """CV balanced accuracy predicting y from X, with a label-shuffle 95% chance line."""
    y = np.asarray(y).astype(int)
    if X.size == 0 or np.abs(X).max() < 1e-20 or len(np.unique(y)) < 2:   # e.g. ablate_spatial lower grad == 0
        return dict(acc=0.5, chance_hi=0.5)
    def cv(yv, rng):
        fo = _folds(yv, k, rng); pr = np.zeros(len(yv)); tt = np.zeros(len(yv))
        for f in range(k):
            te = fo[f]; tri = np.concatenate([fo[j] for j in range(k) if j != f])
            mu = X[tri].mean(0); sd = X[tri].std(0); sd[sd < 1e-8] = 1.0
            Xt = (X[tri] - mu) / sd; Xe = (X[te] - mu) / sd
            ypm = np.where(yv[tri] == yv[tri].max(), 1.0, -1.0)
            p = _ridge(Xt, ypm, Xe)
            pr[te] = np.where(p > 0, yv[tri].max(), yv[tri].min()); tt[te] = yv[te]
        return _bacc(tt, pr)
    rng = np.random.default_rng(seed)
    acc = cv(y, rng)
    perm = np.array([cv(rng.permutation(y), rng) for _ in range(E5B_NPERM)])
    return dict(acc=acc, chance_hi=float(np.percentile(perm, 95)))

_LYRS = {"lower": E5_LOWER, "upper": E5_UPPER}
e5b = {l: {m: {"label": [], "margin": []} for m in E5B_MODES} for l in _LYRS}
e5c_geomX, e5c_label0, e5c_count0 = {}, None, None    # seed-0 lower grads cached for 5.1c (PCA)
for s in range(E5B_SEEDS):
    torch.manual_seed(1000 + s)
    m = DeepRNN(5, E5_NREC, 2, n_layers=2, alpha=E5_ALPHA).to(DEVICE)
    inp, tgt, msk = e5_batch(E5B_TRIALS, E5_DELAY_MAIN, 5000 + s)
    label, margin, count = _cue_vars(inp)
    for mode in E5B_MODES:
        Xl, Xu = _per_trial_grads(m, inp, tgt, msk, mode)
        for lyr, X in (("lower", Xl), ("upper", Xu)):
            e5b[lyr][mode]["label"].append(_decode(X, label, seed=s))
            e5b[lyr][mode]["margin"].append(_decode(X, margin, seed=s))
        if s == 0:
            e5c_geomX[mode] = Xl                      # stash seed-0 lower grads for 5.1c
    if s == 0:
        e5c_label0, e5c_count0 = label, count
    print(f"  seed {s} done (unanimous frac={margin.mean():.2f})")

def _agg(lst, key):
    v = np.array([d[key] for d in lst])
    return float(v.mean()), float(v.std(ddof=1) / np.sqrt(len(v)) if len(v) > 1 else 0.0)

e5b_sum = {}
for lyr in _LYRS:
    e5b_sum[lyr] = {}
    for mode in E5B_MODES:
        e5b_sum[lyr][mode] = {}
        for t in ("label", "margin"):
            mu, se = _agg(e5b[lyr][mode][t], "acc"); chi, _ = _agg(e5b[lyr][mode][t], "chance_hi")
            e5b_sum[lyr][mode][t] = dict(acc=mu, sem=se, chance_hi=chi)
            print(f"  {lyr:5s} {mode:16s} {t:6s}: {mu:.3f}+-{se:.3f} [95% {chi:.3f}]")

# ── figure: 2 panels (lower / upper), grouped bars over {label, margin} ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
_TG = [("label", "binary label\n(readout-sign leak)"), ("margin", "cue margin\n(unanimous vs split)")]
for ax, lyr, ttl in [(axes[0], "lower", "input-adjacent (lower) layer"),
                     (axes[1], "upper", "output-adjacent (top) layer")]:
    x = np.arange(len(_TG)); w = 0.26
    for i, mode in enumerate(E5B_MODES):
        mus = [e5b_sum[lyr][mode][t]["acc"] for t, _ in _TG]
        ers = [e5b_sum[lyr][mode][t]["sem"] for t, _ in _TG]
        ax.bar(x + (i - 1) * w, mus, w, yerr=ers, capsize=4, color=E5B_COLOR[mode],
               label=mode.replace("ablate_", "ablate "))
    chi = np.mean([e5b_sum[lyr][mode][t]["chance_hi"] for mode in E5B_MODES for t, _ in _TG])
    ax.axhline(0.5, color="gray", ls="--", lw=1)
    ax.axhline(chi, color="gray", ls=":", lw=1, label="perm. 95%")
    ax.set_xticks(x); ax.set_xticklabels([lab for _, lab in _TG])
    ax.set_title(ttl); ax.set_ylim(0.4, 1.02)
    if lyr == "lower":
        ax.set_ylabel("cross-validated decode accuracy"); ax.legend(fontsize=8, loc="upper right")
fig.suptitle("Cue information in the per-trial gradient: the temporal carry makes "
             "lower-layer credit cue-specific", fontsize=11)
fig.tight_layout(); save_fig(fig, "exp5_cue_decoding"); plt.show()

with open("results/exp5_cue_decoding.json", "w") as _f:
    json.dump({"seeds": E5B_SEEDS, "trials": E5B_TRIALS, "delay": E5_DELAY_MAIN, "summary": e5b_sum}, _f, indent=2)
print("Saved exp5 cue decoding.")


### 5.1c  Gradient geometry — visualizing cue-type separability (PCA)

The decode-accuracy bars above are a single number per (mode, target). Here we look at the
actual **geometry**: project the seed-0 lower-layer per-trial gradient onto its top-2
principal components (PCA, from-scratch via SVD — no `sklearn`/`umap` dependency) and color
each trial by its **cue type** — the number of rising cues (0–3). The dominant PC1 axis
(~99–100% of variance) is driven by the majority direction (the readout-sign flip
δ=softmax(o)−y) and splits the cloud into two blobs (0–1 rising vs 2–3 rising) regardless of
mode; the finer within-blob ordering by count appears under `full` (and, more weakly,
`ablate_temporal`), and each panel is annotated with the readout-sign-independent **cue-margin**
decode accuracy on the same data. `ablate_spatial`'s lower-layer gradient is identically
zero, so there is nothing to plot.

In [ ]:
# ── 5.1c  Gradient geometry: PCA of the seed-0 lower-layer per-trial gradient ──
print("5.1c gradient geometry (PCA) ...")

def _pca_2d(X):
    """Project (N, D) onto its top-2 principal components via SVD.
    Returns (proj (N,2), explained_variance_ratio (2,)), or None if X is
    degenerate (e.g. ablate_spatial's lower gradient is identically zero)."""
    if X.size == 0 or np.abs(X).max() < 1e-20:
        return None
    Xc = X - X.mean(0, keepdims=True)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    proj = Xc @ Vt[:2].T
    var = S ** 2
    return proj, (var[:2] / var.sum())

# Reuse the seed-0 per-trial gradients + margin decodes cached by 5.1b (saves
# ~1200 single-trial gradient calls + the seed-0 margin decodes). Falls back to
# recomputing if 5.1b hasn't been run in this session, so 5.1c still works alone.
try:
    e5c_geomX; e5c_count0; e5c_label0; e5b
    e5c_X, _count0, _label0 = e5c_geomX, e5c_count0, e5c_label0
    _acc_of = {mode: e5b["lower"][mode]["margin"][0]["acc"] for mode in E5B_MODES}
    print("  reusing seed-0 gradients + decodes from 5.1b (cache hit)")
except NameError:
    print("  5.1b results not in memory - recomputing seed-0 gradients (fallback)")
    torch.manual_seed(1000 + 0)
    _m0 = DeepRNN(5, E5_NREC, 2, n_layers=2, alpha=E5_ALPHA).to(DEVICE)
    _inp0, _tgt0, _msk0 = e5_batch(E5B_TRIALS, E5_DELAY_MAIN, 5000 + 0)
    _label0, _margin0, _count0 = _cue_vars(_inp0)
    e5c_X, _acc_of = {}, {}
    for mode in E5B_MODES:
        Xl, _ = _per_trial_grads(_m0, _inp0, _tgt0, _msk0, mode)
        e5c_X[mode] = Xl
        _acc_of[mode] = _decode(Xl, _margin0, seed=0)["acc"]

_ccolors = {c: plt.cm.viridis(c / E5_NCUES) for c in range(E5_NCUES + 1)}   # 0..N rising cues

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
e5c_data = {}
for ax, mode in zip(axes, E5B_MODES):
    X = e5c_X[mode]
    out = _pca_2d(X)
    mode_lab = mode.replace("ablate_", "ablate ")
    if out is None:
        ax.text(0.5, 0.5, "gradient ≡ 0\n(ablate_spatial removes\nlower-layer credit)",
                ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(mode_lab, fontsize=10)
        e5c_data[mode] = None
    else:
        proj, var = out
        acc = _acc_of[mode]
        for c in range(E5_NCUES + 1):
            sel = _count0 == c
            if not sel.any():
                continue
            ax.scatter(proj[sel, 0], proj[sel, 1], s=14, alpha=0.7,
                       color=_ccolors[c], label=f"{c} rising")
        ax.set_xlabel(f"PC1 ({var[0]*100:.0f}%)")
        ax.set_ylabel(f"PC2 ({var[1]*100:.0f}%)")
        ax.set_title(f"{mode_lab}\ndecode acc (margin) = {acc:.2f}", fontsize=10)
        e5c_data[mode] = dict(proj=proj.tolist(), explained_variance_ratio=var.tolist(),
                              margin_decode_acc=acc)
axes[0].legend(fontsize=8, loc="best", title="rising cues")
fig.suptitle("Geometry of the per-trial lower-layer gradient (PCA, seed 0), colored by "
             "cue type (rising-cue count): the temporal carry makes it cue-specific", fontsize=11)
fig.tight_layout(); save_fig(fig, "exp5_gradient_geometry"); plt.show()

with open("results/exp5_gradient_geometry.json", "w") as _f:
    json.dump({"seed": 0, "modes": E5B_MODES, "count": _count0.tolist(),
               "label": _label0.tolist(), "per_mode": e5c_data}, _f, indent=2)
print("Saved exp5 gradient geometry.")


In [ ]:
# ── 5.2  Adam learning curves: credit quality sets convergence SPEED ──────────
# Trained with Adam, which normalises the per-synapse step size and so removes the
# ablations' gradient-MAGNITUDE deficit, leaving only gradient direction/structure.
# Under fair optimisation every TRAINABLE rule converges to the SAME final accuracy
# on this task, so the credit-assignment difference shows up as convergence SPEED
# (full > ablate_temporal > ablate_spatial), which these curves display. Each method
# trains to its own convergence (patience + min-steps floor). There is no plateau-
# height claim among the trainable rules; the SPEED differences are significance-
# tested in 5.3. The readout-only RESERVOIR (both hidden layers frozen at random
# init; only W_out trains) is trained here too and overlaid as the FLOOR — it does
# NOT converge to the trainable rules' accuracy, so its gap to the ablation curves
# shows how much even a distorted credit signal buys over a pure random reservoir
# (quantified + significance-tested in 5.4).
print(f"5.2 learning curves (D={E5_DELAY_MAIN}, {E5_SEEDS} seeds, Adam lr={E5_LR:.0e}) ...")

def e5_eval(m, delay):
    accs = []
    for e in range(E5_EVAL_REPS):
        ie, te, me = e5_batch(E5_EVAL_N, delay, 90000 + e)
        with torch.no_grad():
            o, _ = m(ie)
        accs.append(hacc(o, te, me))
    return float(np.mean(accs))

e5_methods = [("bptt", "BPTT"), ("full", "deep e-prop (full)"),
              ("ablate_temporal", "ablate temporal"), ("ablate_spatial", "ablate spatial")]

def e5_grads(method, m, inp, tgt, msk):
    if method == "bptt":
        return compute_bptt_gradients(m, inp, tgt, msk, _xent_loss)
    if method == "readout":                       # readout-only reservoir control (5.4):
        g = compute_deep_eprop_gradients(m, inp, tgt, msk, xent_error, mode="full")
        return {k: v for k, v in g.items() if k in ("W_out", "b_out")}   # freeze BOTH layers
    return compute_deep_eprop_gradients(m, inp, tgt, msk, xent_error, mode=method)

# ── Patience-based training with Adam: stop once each method has converged ────
def e5_train(method, seed):
    torch.manual_seed(seed)
    m = DeepRNN(5, E5_NREC, 2, n_layers=2, alpha=E5_ALPHA).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=E5_LR)
    accs, ema, best, since, st = [], None, -1.0, 0, 0
    while True:
        if st % E5_EVAL == 0:
            a = e5_eval(m, E5_DELAY_MAIN); accs.append(a)
            ema = a if ema is None else 0.3 * a + 0.7 * ema
            if ema > best + E5_TOL:
                best, since = ema, 0
            else:
                since += 1
            if (since >= E5_PATIENCE and st >= E5_MIN_STEPS) or st >= E5_MAX_STEPS:
                break
        inp, tgt, msk = e5_batch(E5_BATCH, E5_DELAY_MAIN, 10_000 + seed * 1_000_000 + st)
        g = e5_grads(method, m, inp, tgt, msk)
        opt.zero_grad()
        for pname, p in m.named_parameters():            # custom grads -> Adam step
            p.grad = g.get(pname, torch.zeros_like(p))
        torch.nn.utils.clip_grad_norm_(m.parameters(), E5_GRAD_CLIP)
        opt.step()
        st += 1
    return accs            # ragged: length = #evals until this run converges

def e5_tail_slope(r):      # per-100-step slope over the converged tail (≈0 if flat)
    k = min(E5_TAIL_K, len(r))
    return float(np.polyfit(np.arange(k), np.asarray(r[-k:]), 1)[0]) if k >= 2 else 0.0

def e5_fit_curve(rows):    # ragged per-seed eval curves -> (mean, sem) on a nan-padded grid
    L = max(len(r) for r in rows)
    M = np.full((len(rows), L), np.nan)
    for i, r in enumerate(rows):
        M[i, :len(r)] = r
    return e5_stats(M)

e5_curves, e5_convacc, e5_stops, e5_rows = {}, {}, {}, {}
for meth, lab in e5_methods:
    t0 = time.time()
    rows = [e5_train(meth, s) for s in range(E5_SEEDS)]   # ragged per-seed eval curves
    e5_rows[meth] = rows
    e5_curves[meth]  = e5_fit_curve(rows)                 # nan-aware mean / SEM over seeds
    e5_convacc[meth] = np.array([float(np.mean(r[-E5_TAIL_K:])) for r in rows])  # converged acc
    e5_stops[meth]   = [(len(r) - 1) * E5_EVAL for r in rows]
    slope = float(np.mean([e5_tail_slope(r) for r in rows]))
    print(f"  {lab:22s} conv-acc={e5_convacc[meth].mean():.3f}  "
          f"steps-to-converge≈{int(np.mean(e5_stops[meth]))}  tail-slope/100={slope:+.4f}  "
          f"[{time.time()-t0:.0f}s]")
print("Convergence check: conv-acc should be ~equal across the TRAINABLE rules and "
      "|tail-slope/100|≈0 (raise E5_MAX_STEPS if any still slopes up).")

# ── Readout-only RESERVOIR floor (both hidden layers frozen at random init) ───
# Same schedule/optimiser; kept OUT of e5_methods so the 5.3 speed stats stay on the
# rules that converge to equal accuracy. Overlaid on the figure as the floor and
# reused by 5.4 (so 5.4 does not retrain it).
t0 = time.time()
res_rows = [e5_train("readout", s) for s in range(E5_SEEDS)]
e5_rows["readout"]    = res_rows
e5_curves["readout"]  = e5_fit_curve(res_rows)
e5_convacc["readout"] = np.array([float(np.mean(r[-E5_TAIL_K:])) for r in res_rows])
e5_stops["readout"]   = [(len(r) - 1) * E5_EVAL for r in res_rows]
print(f"  {'readout-only (reservoir)':22s} conv-acc={e5_convacc['readout'].mean():.3f}  "
      f"steps-to-converge≈{int(np.mean(e5_stops['readout']))}  [{time.time()-t0:.0f}s]  "
      f"(floor; excluded from the equal-accuracy / 5.3 speed comparison)")

# ── Figure: Adam curves — trainable rules converge equal (speed differs), ─────
#            with the random-reservoir floor overlaid.
col = {"bptt": "k", "full": "C0", "ablate_temporal": "C3", "ablate_spatial": "C1",
       "readout": "C7"}
e5_plot_methods = e5_methods + [("readout", "readout-only (reservoir)")]
fig, ax = plt.subplots(figsize=(7, 4.5))
for meth, lab in e5_plot_methods:
    mu, er = e5_curves[meth]
    x = np.arange(len(mu)) * E5_EVAL
    ls = "--o" if meth == "readout" else "-o"
    ax.plot(x, mu, ls, ms=3, color=col[meth], label=lab)
    ax.fill_between(x, mu - er, mu + er, color=col[meth], alpha=0.15)
ax.axhline(0.5, color="gray", ls="--", label="chance")
ax.set_xlabel("training step"); ax.set_ylabel("accuracy (held-out)"); ax.set_ylim(0.45, 1.02)
ax.set_title(f"Adam (lr={E5_LR:.0e}): trainable rules converge equal; reservoir floors below")
ax.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "exp5_learning_curves"); plt.show()

# Snapshot the prior shared-LR SGD curves once (before overwrite) for before/after contrast.
import os, shutil
_old, _snap = "results/exp5_learning_curves.json", "results/exp5_learning_curves_sgd.json"
if os.path.exists(_old) and not os.path.exists(_snap):
    shutil.copy(_old, _snap); print(f"Snapshotted prior SGD curves -> {_snap}")

with open(_old, "w") as _f:
    json.dump({"eval_every": E5_EVAL, "optimizer": "adam", "lr": float(E5_LR),
               "curves":      {m: [c[0].tolist(), c[1].tolist()] for m, c in e5_curves.items()},
               "conv_acc":    {m: e5_convacc[m].tolist()          for m, _ in e5_plot_methods},
               "stop_steps":  {m: e5_stops[m]                     for m, _ in e5_plot_methods},
               "per_seed":    {m: e5_rows[m]                      for m, _ in e5_plot_methods}}, _f)
print("Saved Adam exp5 learning curves incl. reservoir floor (+ per-seed curves for 5.3/5.4).")


In [ ]:
# ── 5.3  Speed significance: are the convergence-rate gaps real? ──────────────
# Adam makes every rule converge to the same accuracy, so the credit-assignment
# signal is in SPEED. Two independent, paired (per-seed) tests on the 5.2 curves —
# no extra training. Comparisons: full vs ablate_temporal, full vs ablate_spatial,
# ablate_temporal vs ablate_spatial (Holm-corrected over the 3).
from experiments.stats import paired_report, holm, cluster_perm_test

# Re-run-safe: rebuild per-seed curves from disk if 5.2 isn't in memory.
try:
    e5_rows; e5_methods; col; E5_EVAL
except NameError:
    with open("results/exp5_learning_curves.json") as _f:
        _d = json.load(_f)
    e5_methods = [("bptt", "BPTT"), ("full", "deep e-prop (full)"),
                  ("ablate_temporal", "ablate temporal"), ("ablate_spatial", "ablate spatial")]
    e5_rows = {m: _d["per_seed"][m] for m, _ in e5_methods}
    E5_EVAL = _d["eval_every"]
    col = {"bptt": "k", "full": "C0", "ablate_temporal": "C3", "ablate_spatial": "C1"}

E5_SPEED_COMPS = [("full", "ablate_temporal", "full vs ablate_temporal", "C4"),
                  ("full", "ablate_spatial",  "full vs ablate_spatial",  "C5"),
                  ("ablate_temporal", "ablate_spatial", "temporal vs spatial", "C6")]
_lab = {m: lab for m, lab in e5_methods}

def _stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

# ── (A) Steps-to-threshold ────────────────────────────────────────────────────
def steps_to_threshold(r, theta, eval_every=E5_EVAL):
    """Interpolated training step at which curve r first reaches theta (NaN if never)."""
    r = np.asarray(r, float)
    above = np.where(r >= theta)[0]
    if len(above) == 0:
        return float("nan")
    i = int(above[0])
    if i == 0:
        return 0.0
    y0, y1 = r[i - 1], r[i]
    frac = 0.0 if y1 == y0 else (theta - y0) / (y1 - y0)
    return (i - 1 + min(max(frac, 0.0), 1.0)) * eval_every

def speed_threshold_report(theta):
    s2t = {m: np.array([steps_to_threshold(r, theta) for r in e5_rows[m]])
           for m, _ in e5_methods}
    cens = {m: int(np.isnan(s2t[m]).sum()) for m, _ in e5_methods}
    reports = [(lab, a, b, paired_report(s2t[a], s2t[b])) for a, b, lab, _ in E5_SPEED_COMPS]
    padj = holm([r["p_perm"] for *_, r in reports])
    print(f"\nSteps to reach acc≥{theta:.2f}  (paired sign-flip perm, Holm over {len(reports)}):")
    for m, _ in e5_methods:
        c = f"  ({cens[m]} censored)" if cens[m] else ""
        print(f"    {_lab[m]:22s} mean={np.nanmean(s2t[m]):6.0f} steps{c}")
    print(f"    {'comparison':26s} {'Δsteps':>7} {'dz':>6} {'perm p':>8} {'Holm':>8}")
    for (lab, a, b, r), pa in zip(reports, padj):
        print(f"    {lab:26s} {r['mean_diff']:+7.0f} {r['cohen_dz']:+6.2f} "
              f"{r['p_perm']:8.4f} {pa:8.4f}  {_stars(pa)}")
    return s2t, reports, padj

E5_THETA = 0.9
s2t, e5_spd_reports, e5_spd_padj = speed_threshold_report(E5_THETA)
speed_threshold_report(0.8)        # robustness check at a lower threshold

# Figure A (independent): mean steps-to-threshold per method ± SEM, with brackets.
order = ["bptt", "full", "ablate_temporal", "ablate_spatial"]
short = {"bptt": "BPTT", "full": "full", "ablate_temporal": "ablate\ntemporal",
         "ablate_spatial": "ablate\nspatial"}
means = {m: float(np.nanmean(s2t[m])) for m in order}
sems  = {m: float(np.nanstd(s2t[m], ddof=1) / np.sqrt(np.sum(~np.isnan(s2t[m])))) for m in order}
figA, axA = plt.subplots(figsize=(7, 4.6)); xs = np.arange(len(order))
axA.bar(xs, [means[m] for m in order], yerr=[sems[m] for m in order], capsize=4,
        color=[col[m] for m in order])
axA.set_xticks(xs); axA.set_xticklabels([short[m] for m in order])
axA.set_ylabel(f"training steps to reach acc ≥ {E5_THETA:.2f}")
axA.set_title(f"Convergence speed (Adam): steps to {E5_THETA:.0%} accuracy")
y0 = max(means[m] + sems[m] for m in order)
for k, (a, b, lab, _) in enumerate(E5_SPEED_COMPS):
    xa, xb = order.index(a), order.index(b); y = y0 * (1.06 + 0.10 * k)
    axA.plot([xa, xa, xb, xb], [y - y0 * 0.02, y, y, y - y0 * 0.02], color="k", lw=1)
    axA.text((xa + xb) / 2, y, _stars(e5_spd_padj[k]), ha="center", va="bottom", fontsize=11)
axA.set_ylim(0, y0 * 1.45)
figA.tight_layout(); save_fig(figA, "exp5_speed_threshold"); plt.show()

# ── (B) Cluster-permutation: where over training are the gaps significant? ─────
# Common rectangular window: up to the earliest stop across all methods/seeds.
n_common = min(len(r) for m, _ in e5_methods for r in e5_rows[m])
A = {m: np.array([r[:n_common] for r in e5_rows[m]]) for m, _ in e5_methods}
t_axis = np.arange(n_common) * E5_EVAL

print(f"\nCluster-permutation significant intervals (window 0–{t_axis[-1]} steps, "
      f"pointwise α=0.05, FWER-controlled over time):")
e5_sig_intervals = {}
for a, b, lab, _c in E5_SPEED_COMPS:
    cl = cluster_perm_test(A[a] - A[b])
    sig = [c for c in cl if c["p_cluster"] < 0.05]
    e5_sig_intervals[(a, b)] = sig
    spans = ", ".join(f"[{t_axis[c['t0']]}–{t_axis[c['t1']]}] steps (p={c['p_cluster']:.3f})"
                      for c in sig) or "none"
    print(f"    {lab:26s} {spans}")

# Figure B (independent): mean curves over the window + per-comparison sig bars.
figB, axB = plt.subplots(figsize=(7, 4.8))
for meth, lab in e5_methods:
    axB.plot(t_axis, A[meth].mean(0), "-o", ms=3, color=col[meth], label=lab)
axB.axhline(0.5, color="gray", ls="--", lw=0.8)
axB.set_xlabel("training step"); axB.set_ylabel("accuracy (held-out)")
axB.set_ylim(0.40, 1.02)
axB.set_title("Where convergence-speed gaps are significant (cluster permutation)")
ybar = 0.46
for k, (a, b, lab, cc) in enumerate(E5_SPEED_COMPS):
    y = ybar - 0.018 * k
    for c in e5_sig_intervals[(a, b)]:
        axB.plot([t_axis[c["t0"]], t_axis[c["t1"]]], [y, y], color=cc, lw=4, solid_capstyle="butt")
    axB.plot([], [], color=cc, lw=4, label=f"sig: {lab}")
axB.legend(fontsize=7, ncol=2, loc="lower right")
figB.tight_layout(); save_fig(figB, "exp5_speed_intervals"); plt.show()
print("Saved exp5_speed_threshold + exp5_speed_intervals.")


---
### 5.4 Readout-only reservoir control — is depth credit actually necessary?

A **third control**: freeze *both* recurrent layers at their random init and train
only the readout (a 2-layer random reservoir + linear readout / deep ESN). It is
the natural floor for the depth ablation:

- `ablate_spatial` — layer 0 frozen (grad = 0), **layer 1 trained**;
- `ablate_temporal` — cross-layer temporal carry off, **both layers trained**;
- `readout-only`  — layer 0 **and** layer 1 frozen.

The reservoir is trained on the same schedule in **5.2** and overlaid on the
learning-curve figure, so you can read the gap between each credit-carrying rule
(including *both* ablations) and the pure-reservoir floor directly off the curves.
Here we reuse those runs for the quantitative comparison:

- `ablate_spatial − readout-only` isolates *what training the top layer buys over a
  random reservoir*;
- `ablate_temporal − readout-only` isolates *what the surviving depth credit buys*.

If a rule ≫ reservoir, its learning-curve separation is a real credit effect; if
`ablate_spatial ≈ readout-only`, the trained top layer does not rescue the task from
a random lower layer and depth credit is essential. (Reuses 5.2 — run 5.2 first.)


In [ ]:
# ── 5.4  Readout-only reservoir control: is depth credit actually necessary? ──
# Both recurrent layers stay FROZEN at random init; only W_out/b_out train (the
# e-prop readout gradient is exact). This is a 2-layer random reservoir + linear
# readout — the natural FLOOR for ablate_spatial (which freezes layer 0 but TRAINS
# layer 1). (ablate_spatial - readout-only) = what training the top layer buys;
# (ablate_temporal - readout-only) = what the surviving depth credit buys.
# The reservoir is trained in 5.2 (overlaid on the learning-curve figure); here we
# reuse those runs and just do the quantitative comparison + paired significance.
print("5.4 readout-only reservoir control ...")

try:                    # reuses 5.2's per-seed converged accuracies + trainer
    e5_train; e5_convacc; e5_curves; e5_rows; col; E5_EVAL; E5_TAIL_K
except NameError:
    raise RuntimeError("Run 5.2 (the Adam learning-curve cell) before 5.4 — it "
                       "defines e5_train, e5_convacc, e5_curves and the reservoir run.")

# Reuse the reservoir trained in 5.2 (same schedule); fall back to training it here
# only if 5.2 predates this change and didn't produce it.
if "readout" in e5_convacc:
    res_convacc = e5_convacc["readout"]
    res_mu, res_sem = e5_curves["readout"]
    res_rows = e5_rows["readout"]
    print(f"  readout-only (reservoir) conv-acc={res_convacc.mean():.3f} (reused from 5.2)")
else:
    _t0 = time.time()
    res_rows = [e5_train("readout", s) for s in range(E5_SEEDS)]
    res_convacc = np.array([float(np.mean(r[-E5_TAIL_K:])) for r in res_rows])
    _Lr = max(len(r) for r in res_rows)
    _Mr = np.full((len(res_rows), _Lr), np.nan)
    for i, r in enumerate(res_rows):
        _Mr[i, :len(r)] = r
    res_mu, res_sem = e5_stats(_Mr)
    print(f"  readout-only (reservoir) conv-acc={res_convacc.mean():.3f}  [{time.time()-_t0:.0f}s]")
col.setdefault("readout", "C7")

# ── Converged-accuracy bars: trainable rules vs the reservoir floor ───────────
bar_methods = [("bptt", "BPTT"), ("full", "full"), ("ablate_temporal", "ablate\ntemporal"),
               ("ablate_spatial", "ablate\nspatial"), ("readout", "readout-only\n(reservoir)")]
acc_of = {m: (res_convacc if m == "readout" else e5_convacc[m]) for m, _ in bar_methods}
means = {m: float(np.mean(acc_of[m])) for m, _ in bar_methods}
sems  = {m: float(np.std(acc_of[m], ddof=1) / np.sqrt(len(acc_of[m]))) for m, _ in bar_methods}

figc, axc = plt.subplots(figsize=(7.5, 4.6)); xs = np.arange(len(bar_methods))
axc.bar(xs, [means[m] for m, _ in bar_methods], yerr=[sems[m] for m, _ in bar_methods],
        capsize=4, color=[col[m] for m, _ in bar_methods])
axc.set_xticks(xs); axc.set_xticklabels([lab for _, lab in bar_methods])
axc.set_ylabel("converged accuracy (held-out)"); axc.axhline(0.5, color="gray", ls="--", label="chance")
axc.set_ylim(0.45, 1.05); axc.legend(fontsize=8)
axc.set_title("Depth ablation vs a random reservoir: does training the top layer rescue it?")
figc.tight_layout(); save_fig(figc, "exp5_reservoir_control"); plt.show()

# ── Learning-curve overlay: reservoir floor against the trainable rules ───────
figd, axd = plt.subplots(figsize=(7, 4.5))
for meth, lab in [("bptt", "BPTT"), ("full", "full"),
                  ("ablate_temporal", "ablate temporal"), ("ablate_spatial", "ablate spatial")]:
    mu, er = e5_curves[meth]; x = np.arange(len(mu)) * E5_EVAL
    axd.plot(x, mu, "-o", ms=3, color=col[meth], label=lab)
    axd.fill_between(x, mu - er, mu + er, color=col[meth], alpha=0.12)
xr = np.arange(len(res_mu)) * E5_EVAL
axd.plot(xr, res_mu, "--o", ms=3, color=col["readout"], label="readout-only (reservoir)")
axd.fill_between(xr, res_mu - res_sem, res_mu + res_sem, color=col["readout"], alpha=0.12)
axd.axhline(0.5, color="gray", ls="--", label="chance")
axd.set_xlabel("training step"); axd.set_ylabel("accuracy (held-out)"); axd.set_ylim(0.45, 1.02)
axd.legend(fontsize=8); axd.set_title("Reservoir floor vs trainable-hidden rules")
figd.tight_layout(); save_fig(figd, "exp5_reservoir_curves"); plt.show()

# ── Paired significance: each trainable rule vs the reservoir floor ──────────
# Does every credit-carrying rule (incl. BOTH ablations) sit significantly above a
# pure random reservoir? full/ablate_temporal >> reservoir and ablate_spatial >>
# reservoir all indicate a real gap; ablate_spatial ~= reservoir would instead mean
# the frozen lower layer is not rescued by training the top (depth credit essential).
from experiments.stats import paired_report, holm
comps = [("full",            "full − reservoir"),
         ("ablate_temporal", "ablate_temporal − reservoir"),
         ("ablate_spatial",  "ablate_spatial − reservoir")]
reports = [(lab, paired_report(e5_convacc[m], res_convacc)) for m, lab in comps]
padj = holm([r["p_perm"] for _, r in reports])
print(f"  {'comparison':30s} {'meanD':>8} {'dz':>7} {'perm p':>8} {'Holm p':>8}  sig")
for (lab, r), pa in zip(reports, padj):
    sig = "***" if pa < 0.001 else "**" if pa < 0.01 else "*" if pa < 0.05 else "ns"
    print(f"  {lab.replace(chr(8722), '-'):30s} {r['mean_diff']:+8.3f} {r['cohen_dz']:+7.2f} {r['p_perm']:8.4f} {pa:8.4f}  {sig}")

with open("results/exp5_reservoir_control.json", "w") as _f:
    json.dump({"delay": E5_DELAY_MAIN, "n_seeds": E5_SEEDS,
               "conv_acc": {**{m: acc_of[m].tolist() for m, _ in bar_methods if m != "readout"},
                            "readout": res_convacc.tolist()},
               "means": means, "sems": sems,
               "res_curve": [res_mu.tolist(), res_sem.tolist()],
               "comparisons": [{"comparison": lab, **r, "p_holm": float(pa)}
                               for (lab, r), pa in zip(reports, padj)],
               "per_seed_readout": res_rows}, _f)
print("Interpretation: every trainable rule (incl. both ablations) >> reservoir means")
print("even a distorted credit signal beats a pure random reservoir; the ablation↔reservoir")
print("gaps are the learning-curve separations you now see overlaid in 5.2.")
print("Saved exp5_reservoir_control + exp5_reservoir_curves.")


---
## 6. Summary and conclusions

### Results at a glance

| Experiment | Key finding | Confirmed? |
|-----------|-------------|:--:|
| **1 — Single-layer, tanh** | e-prop ≈ d=0 (gap < 0.004): the diagonal tanh carry is ≈ 0.005, so the trace barely persists; both reach high cosine with BPTT at short delay | ✅ |
| **2 — Leaky RNN (α sweep)** | The e-prop > d=0 gap grows as α decreases (stronger leak, longer trace): e-prop captures the (1−α) temporal carry that d=0 discards | ✅ |
| **3 — Time + depth (main result)** | Full deep e-prop tracks BPTT for BOTH layers; `ablate_spatial`→lower-layer credit = 0, `ablate_temporal`→lower credit degraded (≈90% of it flows through the cross-layer trace εᶻ); learning: BPTT ≥ full e-prop > both controls, gap widening with delay | ✅ |

### Main conclusions

**1. For vanilla tanh RNNs, e-prop ≈ d=0.** The diagonal temporal carry (≈ 0.005/step) is so small that the eligibility trace resets almost immediately, so dropping it (the d=0 rule) barely changes the gradient. (Consistent with the Shalev-Merin 2026 analysis for randomly-initialised networks.)

**2. Leaky neurons create a meaningful temporal trace.** With carry ≈ (1−α)/step the trace survives ≈ 1/(1−α) steps, so e-prop assigns real credit across a silent delay; the e-prop vs d=0 gap grows with both delay and leak strength.

**3. (Main result) Deep e-prop assigns credit across time AND depth simultaneously.** On a hierarchical *classify-then-count* task — the lower layer must learn a temporal feature (depth) whose per-cue output the top layer accumulates over a delay (time) — full deep e-prop aligns with BPTT for *both* layers and trains the network nearly as well as BPTT. The two controls on the top-layer trace `εᶻ = (∂z/∂h)·εʰ + (∂z/∂zₜ₋₁)·εᶻₜ₋₁` each break it: removing the **depth** path (∂z/∂h = 0) zeros the lower-layer gradient; removing the **cross-layer temporal** path (∂z/∂zₜ₋₁ = 0) degrades it (≈90% of lower-layer credit flows through `εᶻ`). Learning ordering is **BPTT ≥ full deep e-prop > both controls**, with the gap widening as the delay grows.

### Limitations and future work

- All experiments use **random initialisation**; after training, recurrent self-weights may grow and change the effective temporal carry.
- The exact deep e-prop cross-layer trace costs **O(L²·B·n³) memory**, which bounds the depth/width at which it is practical; Experiment 3 uses a fixed 2-layer width and one task.
- BPTT saturates the Experiment-3 task (≈100%); a harder task would further sharpen the BPTT-vs-e-prop gap.

All figures (PDF + SVG) are written to `results/` (each Full-rerun cell overwrites its own files there; nothing is wiped up front — see Setup) and `figures/`. Run the cell below to push outputs to GitHub.

---
## Push results to GitHub

Run the cell below to commit all generated figures and push them to the repo.
You need a GitHub **Personal Access Token** with `repo` scope stored as a Colab secret
named `GITHUB_TOKEN` (Colab sidebar → 🔑 Secrets).

In [ ]:
import subprocess

# ── Configure git identity ───────────────────────────────────────────────────
subprocess.run(['git', 'config', 'user.name',  'Simon Peter'],        check=True)
subprocess.run(['git', 'config', 'user.email', 'go75wiv@mytum.de'],   check=True)

# ── Authenticate via GitHub token stored in Colab Secrets ───────────────────
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')   # set this in the 🔑 Secrets panel
subprocess.run([
    'git', 'remote', 'set-url', 'origin',
    f'https://{token}@github.com/simonpeter02/e-prop-in-deep-networks.git'
], check=True)

# ── Stage all generated figures ──────────────────────────────────────────────
subprocess.run(['git', 'add', '-A', 'results/', 'figures/'], check=True)  # -A stages deletions → results/ fully rewritten

# Check if there is anything new to commit
status = subprocess.run(['git', 'status', '--porcelain'], capture_output=True, text=True)
if not status.stdout.strip():
    print("Nothing new to commit — all figures already up to date.")
else:
    print("Staging:\n" + status.stdout)
    subprocess.run([
        'git', 'commit', '-m',
        'Add generated figures from Colab run\n\nCo-Authored-By: Claude Opus 4.8 (1M context) <noreply@anthropic.com>'
    ], check=True)
    subprocess.run(['git', 'push', 'origin', 'main'], check=True)
    print("\nPushed successfully.")